# Day 062 — Exercise 5: Full Test Suite

A **test suite** covers the core paths through an API: the happy path (things that should work), validation errors (things that should be rejected), and state transitions (create → read → delete).

Good test isolation means each test gets a **fresh app instance** — no shared state between tests. In pytest, a `@pytest.fixture` on the `client` does this automatically. Here, `_fresh()` returns a new client for each test call.

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from starlette.testclient import TestClient

def build_item_api() -> FastAPI:
    """Provided: simple CRUD API.

    GET /items                    → {"items": [...]}
    POST /items {name, price}     → 201 {id, name, price}
    GET /items/{id}               → {id, name, price} | 404
    DELETE /items/{id}            → 204 | 404
    """
    app = FastAPI()
    _db: dict = {}
    _nxt = {"id": 1}

    class _Item(BaseModel):
        name:  str   = Field(min_length=1)
        price: float = Field(gt=0)

    @app.get("/items")
    def list_items():
        return {"items": list(_db.values())}

    @app.post("/items", status_code=201)
    def create(item: _Item):
        iid = _nxt["id"]; _nxt["id"] += 1
        _db[iid] = {"id": iid, **item.model_dump()}
        return _db[iid]

    @app.get("/items/{iid}")
    def get(iid: int):
        if iid not in _db: raise HTTPException(404, "Not found")
        return _db[iid]

    @app.delete("/items/{iid}", status_code=204)
    def delete(iid: int):
        if iid not in _db: raise HTTPException(404, "Not found")
        del _db[iid]

    return app

def _fresh():
    """Return a fresh TestClient for each test (isolated state)."""
    return TestClient(build_item_api(), raise_server_exceptions=False)


In [ ]:
# no extra imports needed


## Task

Implement four test functions for `build_item_api()`. Each takes a `client` argument.

| Test | What to check |
|------|---------------|
| `test_list_empty` | GET /items → 200, `items == []` |
| `test_create_item` | POST /items → 201, response has `id`, `name`, `price` |
| `test_get_not_found` | GET /items/999 → 404 |
| `test_create_then_delete` | POST → DELETE (204) → GET → 404 |

## Your Implementation

In [ ]:
def test_list_empty(client) -> None:
    """GET /items on a fresh app returns an empty list."""
    # TODO: GET /items, assert 200, assert items == []
    raise NotImplementedError

def test_create_item(client) -> None:
    """POST /items with valid data returns 201 with id, name, price."""
    # TODO: POST /items, assert 201, check id/name/price in response
    raise NotImplementedError

def test_get_not_found(client) -> None:
    """GET /items/999 on a fresh app returns 404."""
    # TODO: GET /items/999, assert 404
    raise NotImplementedError

def test_create_then_delete(client) -> None:
    """Create an item, delete it, confirm it's gone (404 on GET)."""
    # TODO: POST to create, DELETE by id, GET to confirm 404
    raise NotImplementedError


In [ ]:
def test_list_empty(client) -> None:
    r = client.get("/items")
    assert r.status_code == 200
    assert r.json()["items"] == []

def test_create_item(client) -> None:
    r = client.post("/items", json={"name": "Widget", "price": 9.99})
    assert r.status_code == 201
    data = r.json()
    assert data["name"] == "Widget"
    assert data["price"] == 9.99
    assert "id" in data

def test_get_not_found(client) -> None:
    r = client.get("/items/999")
    assert r.status_code == 404

def test_create_then_delete(client) -> None:
    created = client.post("/items", json={"name": "Thing", "price": 1.0}).json()
    r_del = client.delete(f"/items/{created['id']}")
    assert r_del.status_code == 204
    r_get = client.get(f"/items/{created['id']}")
    assert r_get.status_code == 404


## Automated checks

In [ ]:
score, total = 0, 4
try:
    test_list_empty(_fresh())
    score += 1; print("\u2705 test_list_empty passes")
except NotImplementedError:
    print("\u274c test_list_empty not implemented")
except Exception as e:
    print(f"\u274c test_list_empty: {e}")

try:
    test_create_item(_fresh())
    score += 1; print("\u2705 test_create_item passes")
except NotImplementedError:
    print("\u274c test_create_item not implemented")
except Exception as e:
    print(f"\u274c test_create_item: {e}")

try:
    test_get_not_found(_fresh())
    score += 1; print("\u2705 test_get_not_found passes")
except NotImplementedError:
    print("\u274c test_get_not_found not implemented")
except Exception as e:
    print(f"\u274c test_get_not_found: {e}")

try:
    test_create_then_delete(_fresh())
    score += 1; print("\u2705 test_create_then_delete passes")
except NotImplementedError:
    print("\u274c test_create_then_delete not implemented")
except Exception as e:
    print(f"\u274c test_create_then_delete: {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def test_list_empty(client) -> None:
    r = client.get("/items")
    assert r.status_code == 200
    assert r.json()["items"] == []

def test_create_item(client) -> None:
    r = client.post("/items", json={"name": "Widget", "price": 9.99})
    assert r.status_code == 201
    data = r.json()
    assert data["name"] == "Widget"
    assert data["price"] == 9.99
    assert "id" in data

def test_get_not_found(client) -> None:
    r = client.get("/items/999")
    assert r.status_code == 404

def test_create_then_delete(client) -> None:
    created = client.post("/items", json={"name": "Thing", "price": 1.0}).json()
    r_del = client.delete(f"/items/{created['id']}")
    assert r_del.status_code == 204
    r_get = client.get(f"/items/{created['id']}")
    assert r_get.status_code == 404
```

**Why a fresh client per test?** Without isolation, test order matters — `test_list_empty` would fail if `test_create_item` ran first and left an item in the store. pytest fixtures solve this automatically. Here, `_fresh()` is the manual equivalent.

</details>